# 20260720_EDA_2018_전국_시도분리정제
- 작성자: 이정연
- 이슈 #19 참고

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    SUBTOTAL_LABEL_PATTERN,
    UNIT_NOTATION_PATTERN,
    assign_labels,
    backfill_major_category_from_medium,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [2]:
YEAR = 2018
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2018년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE  # Table 1 시트가 같은 파일 안에 있음

ZERO_BUDGET_TOKENS = ("-",)
# 2018은 시도별 단위·소계 행과 페이지마다 반복되는 시행계획 머리글이 있음
REPEATED_PLAN_HEADER_PATTERN = rf"^\s*{YEAR}년도\s+지방자치단체\s+시행계획"
EXTRA_HEADER_PATTERNS = (
    UNIT_NOTATION_PATTERN,
    SUBTOTAL_LABEL_PATTERN,
    REPEATED_PLAN_HEADER_PATTERN,
)
GWANGJU_AGGREGATE_LABELS = {"공통사업+자체사업", "공통사업 (저출산+고령사회)"}
GWANGJU_LOWER_SUBTOTAL_PATTERN = r"^\s*(?:\d+-\d+|\d+\.)\s*"
DAEJEON_MAJOR_LABEL_PATTERN = r"^\s*\[\s*(?:공\s*통|자\s*체)\s*사\s*업\s*\]"
DAEJEON_MEDIUM_LABEL_PATTERN = r"^[Ⅰ-Ⅿ]"
DAEJEON_LOWER_SUBTOTAL_PATTERN = r"\(?\s*\d+\s*개\s*과제\s*\)\s*$"
# 2017은 사업분류재정구분이 계/국비/지방비 등 재원 구분이라 이중계상 위험이 있음 (이슈 #26)
NEEDS_FUNDING_SOURCE_CLEANUP = True
# 원본 표에서 추출 오류임을 직접 확인한 삭제 대상만 지정한다. 현재 확인된 대상 없음.
CONFIRMED_DUPLICATE_ROWS = ()
QA_TOLERANCE = 0
# 지역별 분류와 소계 QA를 확정한 뒤 True로 변경한다.
RUN_LLM = True

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"

## 2. 데이터 로드와 기본 확인

In [3]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", df_raw["지역"].nunique())
display(df_raw.head())

데이터 크기: (8291, 12)
지역 수: 17


,지역,세부사업명,사업분류재정구분,2018년 예산,2017년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,(단위 : 백만원),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,서울,총 계,NaN,5912212,5145633,766579,14.9,NaN,1.0,2.0,4.0,데이터행
2,서울,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,5.0,구분행사업명 연속 후보
3,서울,공통사업 합계,계,5082713,4413465,669248,15.2,NaN,1.0,2.0,6.0,데이터행
4,서울,공통사업 합계,국비,2999506,2583763,415743,16.1,NaN,1.0,2.0,7.0,데이터행


## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [4]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    # 광주 원본의 '지방비' 51행은 첫 글자가 잘린 '방비'로 추출됐다.
    is_gwangju_truncated_local = df_raw["지역"].eq("광주") & df_raw["사업분류재정구분"].astype(
        "string"
    ).str.strip().eq("방비")
    print("광주 '방비' → '지방비' 보정:", int(is_gwangju_truncated_local.sum()))
    assert is_gwangju_truncated_local.sum() == 51
    df_raw.loc[is_gwangju_truncated_local, "사업분류재정구분"] = "지방비"

    # 셀 분리 추출로 바로 붙은 완전 동일 행만 중복으로 확정한다.
    # 사업명과 예산만 같은 정상 사업은 제거하지 않는다.
    duplicate_compare_cols = [column for column in df_raw.columns if column != "원본행"]
    row_fingerprint = pd.util.hash_pandas_object(df_raw[duplicate_compare_cols], index=False)
    is_adjacent_exact_duplicate = row_fingerprint.eq(row_fingerprint.shift())
    adjacent_duplicate_rows = df_raw.loc[is_adjacent_exact_duplicate, "원본행"].tolist()
    confirmed_duplicate_rows = set(CONFIRMED_DUPLICATE_ROWS) | set(adjacent_duplicate_rows)
    print("연속 완전 중복 제거 후보:", len(adjacent_duplicate_rows))
    display(
        df_raw.loc[is_adjacent_exact_duplicate]
        .groupby("지역")
        .size()
        .rename("제거_후보_행수")
        .to_frame()
    )
    df_raw = drop_exact_duplicate_rows(df_raw, confirmed_duplicate_rows=confirmed_duplicate_rows)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# 페이지마다 반복된 시행계획 머리글이 세부사업에 포함되지 않았는지 검증한다.
is_repeated_plan_header = (
    df_raw["세부사업명"]
    .astype("string")
    .str.contains(REPEATED_PLAN_HEADER_PATTERN, regex=True, na=False)
)
print("반복 시행계획 머리글:", int(is_repeated_plan_header.sum()))
display(
    df_raw.loc[is_repeated_plan_header]
    .groupby(["지역", "사업행구분"])
    .size()
    .rename("행수")
    .to_frame()
)
assert is_repeated_plan_header.sum() == 205
assert df_raw.loc[is_repeated_plan_header, "사업행구분"].eq("헤더반복").all()

# 광주의 공통·자체사업 총괄 행은 개별 세부사업이 아니므로 합계에서 제외한다.
is_gwangju_aggregate = df_raw["지역"].eq("광주") & df_raw["세부사업명"].isin(
    GWANGJU_AGGREGATE_LABELS
)
df_raw.loc[is_gwangju_aggregate, "사업행구분"] = "헤더반복"

# 광주 자체사업의 대응기반 제목은 접미사가 없어 기본 중분류 패턴에서 누락된다.
is_gwangju_self_response_medium = df_raw["지역"].eq("광주") & df_raw["세부사업명"].astype(
    "string"
).str.strip().eq("3. 저출산·고령사회 대응기반 강화")
df_raw.loc[is_gwangju_self_response_medium, "세부사업명"] = (
    "3. 저출산·고령사회 대응기반 강화(자체사업)"
)
df_raw.loc[is_gwangju_self_response_medium, "사업행구분"] = "중분류_소계"

# 광주의 번호형 제목은 하위 사업 합계를 다시 적은 행이므로 leaf 합계에서 제외한다.
gwangju_detail_name = df_raw["세부사업명"].astype("string").str.strip()
is_gwangju_lower_subtotal = (
    df_raw["지역"].eq("광주")
    & df_raw["사업행구분"].eq("세부사업")
    & gwangju_detail_name.str.match(GWANGJU_LOWER_SUBTOTAL_PATTERN, na=False)
)
print("광주 하위 소계 제외:", int(is_gwangju_lower_subtotal.sum()))
assert is_gwangju_lower_subtotal.sum() == 19
df_raw.loc[is_gwangju_lower_subtotal, "사업행구분"] = "헤더반복"

# 서울 자체사업 구간의 중분류 제목은 '(자체사업)' 접미사가 없어 기본 분류에서 누락된다.
SEOUL_SELF_MEDIUM_LABELS = {
    "1. 저출산 대책": "1. 저출산 대책(자체사업)",
    "2. 고령화 대책": "2. 고령화 대책(자체사업)",
}
is_seoul_self_medium = df_raw["지역"].eq("서울") & df_raw["세부사업명"].astype(
    "string"
).str.strip().isin(SEOUL_SELF_MEDIUM_LABELS)
df_raw.loc[is_seoul_self_medium, "세부사업명"] = (
    df_raw.loc[is_seoul_self_medium, "세부사업명"]
    .astype("string")
    .str.strip()
    .map(SEOUL_SELF_MEDIUM_LABELS)
)
df_raw.loc[is_seoul_self_medium, "사업행구분"] = "중분류_소계"

# 대전은 'N. 저출산/고령사회/대응기반'이 중분류이고 'N)' 제목은 하위 소계다.
is_daejeon = df_raw["지역"].eq("대전")
daejeon_detail_name = df_raw["세부사업명"].astype("string").str.strip()
is_daejeon_medium = is_daejeon & daejeon_detail_name.str.contains(
    r"^[1-3]\.[\s\S]*\(\s*\d+\s*개\s*과제\s*\)\s*$",
    regex=True,
    na=False,
)
is_daejeon_lower_subtotal = (
    is_daejeon
    & ~is_daejeon_medium
    & ~daejeon_detail_name.str.match(r"^[Ⅰ-Ⅿ]", na=False)
    & daejeon_detail_name.str.contains(DAEJEON_LOWER_SUBTOTAL_PATTERN, regex=True, na=False)
)
is_daejeon_total = is_daejeon & daejeon_detail_name.str.match(r"^총\s*계", na=False)
df_raw.loc[is_daejeon_medium, "사업행구분"] = "중분류_소계"
df_raw.loc[is_daejeon_lower_subtotal | is_daejeon_total, "사업행구분"] = "헤더반복"

# 울산 자체사업의 마지막 주제 제목은 접미사가 없어 기본 중분류 패턴에서 누락된다.
is_ulsan_self_medium = df_raw["지역"].eq("울산") & df_raw["세부사업명"].astype(
    "string"
).str.strip().eq("3. 대응기반 강화")
df_raw.loc[is_ulsan_self_medium, "세부사업명"] = "3. 대응기반 강화(자체사업)"
df_raw.loc[is_ulsan_self_medium, "사업행구분"] = "중분류_소계"

df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# 경남은 마지막 공통사업 중분류의 접미사가 원본에서 '(자체사업)'으로 잘못 표기됐다.
# 대분류 역채움 전에 고쳐야 해당 구간이 자체사업으로 잘못 분류되지 않는다.
is_gyeongnam_common_medium_typo = df_labeled["지역"].eq("경남") & df_labeled["중분류"].eq(
    "3. 대응기반 강화(자체사업)"
)
df_labeled.loc[is_gyeongnam_common_medium_typo, "중분류"] = "3. 대응기반 강화(공통사업)"

# 로마자 대분류 행이 없는 경남·광주는 중분류의 공통/자체 접미사로 대분류를 역채움한다.
df_labeled = backfill_major_category_from_medium(df_labeled)

# 대전 대·중분류에서 과제 수를 제거하고 다른 지역과 같은 공통/자체 표기로 맞춘다.
is_daejeon_labeled = df_labeled["지역"].eq("대전")
daejeon_major = clean_text(df_labeled.loc[is_daejeon_labeled, "대분류"]).str.replace(
    r"\s*\(\s*\d+\s*개\s*과제\s*\)\s*$", "", regex=True
)
df_labeled.loc[is_daejeon_labeled, "대분류"] = daejeon_major
daejeon_medium = clean_text(df_labeled.loc[is_daejeon_labeled, "중분류"]).str.replace(
    r"\s*\(\s*\d+\s*개\s*과제\s*\)\s*$", "", regex=True
)
daejeon_suffix = df_labeled.loc[is_daejeon_labeled, "대분류"].map(
    {"Ⅰ. 공통사업": "(공통사업)", "Ⅱ. 지자체 자체사업": "(자체사업)"}
)
df_labeled.loc[is_daejeon_labeled, "중분류"] = daejeon_medium + daejeon_suffix

# 역채움된 대분류를 다른 지역과 동일한 최종 표기로 통일한다.
MAJOR_CATEGORY_NORMALIZATION = {
    "공통사업": "Ⅰ. 공통사업",
    "자체사업": "Ⅱ. 지자체 자체사업",
    "Ⅱ. 자체사업": "Ⅱ. 지자체 자체사업",
}
df_labeled["대분류"] = df_labeled["대분류"].replace(MAJOR_CATEGORY_NORMALIZATION)

# 경기는 자체사업이 두 구간으로 나뉘며 동일 중분류의 띄어쓰기가 서로 다르다.
# 두 구간을 같은 분석 분류로 집계할 수 있도록 라벨만 통일한다.
GYEONGGI_MEDIUM_CATEGORY_NORMALIZATION = {
    "3. 대응기반강화(자체사업)": "3. 대응기반 강화(자체사업)",
}
is_gyeonggi_labeled = df_labeled["지역"].eq("경기")
df_labeled.loc[is_gyeonggi_labeled, "중분류"] = df_labeled.loc[
    is_gyeonggi_labeled, "중분류"
].replace(GYEONGGI_MEDIUM_CATEGORY_NORMALIZATION)
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

광주 '방비' → '지방비' 보정: 51
연속 완전 중복 제거 후보: 0


,제거_후보_행수
지역,


재원구분(계/국비/지방비) 정리 후 행 수: 5998
반복 시행계획 머리글: 205


,,행수
지역,사업행구분,
강원,헤더반복,4
경기,헤더반복,31
경남,헤더반복,5
경북,헤더반복,17
광주,헤더반복,14
대구,헤더반복,10
대전,헤더반복,11
부산,헤더반복,18
서울,헤더반복,8


광주 하위 소계 제외: 19
사업행구분
세부사업      5472
헤더반복       399
중분류_소계      96
대분류_소계      31
Name: count, dtype: int64
세부사업 행 수: 5472


## 4. 텍스트 정제와 예산 재계산

In [5]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

# 원본의 계/국비/지방비는 재원구분에 보존하고, 최종 분류는 대분류에서 생성한다.
df_leaf["재원구분"] = clean_text(df_leaf["재원구분"])
df_leaf["사업분류재정구분"] = pd.Series(pd.NA, index=df_leaf.index, dtype="string")
is_common_leaf = df_leaf["대분류"].str.contains("공통", na=False)
is_self_leaf = df_leaf["대분류"].str.contains("자체", na=False)
df_leaf.loc[is_common_leaf, "사업분류재정구분"] = "공통"
df_leaf.loc[is_self_leaf, "사업분류재정구분"] = "자체"

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head(20))

,지역,세부사업명,당해예산,전년도예산,증감액,증감율
1085,강원,다함께 돌봄센터 설치,100.0,0.0,100.0,<NA>
1086,강원,아이돌봄 서비스 지원,8067.0,7310.0,757.0,10.4
1087,강원,농촌총각 결혼지원,53.0,60.0,-7.0,-11.7
1088,강원,농어업인 고교생자녀 학자금 지원,1520.0,1520.0,0.0,0.0
1089,강원,여성농업인 농가도우미 지원,120.0,75.0,45.0,60.0
1090,강원,맞춤형 주거급여 지원,46690.0,40381.0,6309.0,15.6
1091,강원,저출산 극복 인식개선 프로그램 지원,34.0,34.0,0.0,0.0
1092,강원,지역아동센터 운영,10451.0,9750.0,701.0,7.2
1093,강원,우수 지역아동센터 지원,395.0,387.0,8.0,2.1
1094,강원,지역아동센터 특수목적형 운영,329.0,330.0,-1.0,-0.3


In [6]:
# 지역마다 유니크 대분류가 몇개고 종류가 뭔지 확인
category_by_region = df_leaf.groupby("지역").agg(
    대분류_개수=("대분류", "nunique"),
    대분류_종류=("대분류", lambda s: sorted(s.dropna().unique())),
    중분류_개수=("중분류", "nunique"),
    중분류_종류=("중분류", lambda s: sorted(s.dropna().unique())),
)

category_by_region

,대분류_개수,대분류_종류,중분류_개수,중분류_종류
지역,,,,
강원,1,[Ⅰ. 공통사업],3,"[1. 저출산 대책(공통사업), 2. 고령화 대책(공통사업), 3. 저출산 고령사회 대응기반 강화(공통사업)]"
경기,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",6,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령화 대책(공통사업), 2. 고령화 대책(자체사업), 3. 대응기반 강화(공통사업), 3. 대응기반 강화(자체사업)]"
경남,1,[Ⅰ. 공통사업],3,"[고령화 대책, 대응기반 강화, 저출산 대책]"
경북,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",6,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령화 대책(공통사업), 2. 고령화 대책(자체사업), 3. 대응기반 강화(공통사업), 3. 대응기반 강화(자체사업)]"
광주,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",5,"[고령화 대책, 저출산 고령사회 대응기반 강화, 저출산·고령사회 대응기반 강화, 저출산대책, 저출산분야]"
대구,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",6,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령화 대책(공통사업), 2. 고령화 대책(자체사업), 3. 대응기반 강화(공통사업), 3. 대응기반 강화(자체사업)]"
대전,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",6,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령사회 대책(공통사업), 2. 고령사회 대책(자체사업), 3. 저출산·고령화 대응기반 강화(공통사업), 3. 저출산·고령화 대응기반 강화(자체사업)]"
부산,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",6,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령화 대책(공통사업), 2. 고령화 대책(자체사업), 3. 대응기반 강화(공통사업), 3. 대응기반 강화(자체사업)]"
서울,2,"[Ⅰ. 공통사업, Ⅱ. 지자체 자체사업]",5,"[1. 저출산 대책(공통사업), 1. 저출산 대책(자체사업), 2. 고령화 대책(공통사업), 2. 고령화 대책(자체사업), 3. 대응기반 강화(자체사업)]"


In [7]:
major = df_leaf["대분류"].astype("string")
medium = df_leaf["중분류"].astype("string")

category_qa = df_leaf.groupby("지역").agg(
    leaf_개수=("원본행", "size"),
    대분류_개수=("대분류", "nunique"),
    대분류_결측=("대분류", lambda s: s.isna().sum()),
    중분류_개수=("중분류", "nunique"),
    중분류_결측=("중분류", lambda s: s.isna().sum()),
)

mismatch = (major.str.contains("공통", na=False) & medium.str.contains("자체", na=False)) | (
    major.str.contains("자체", na=False) & medium.str.contains("공통", na=False)
)

category_qa["공통자체_불일치"] = mismatch.groupby(df_leaf["지역"]).sum()

category_qa

,leaf_개수,대분류_개수,대분류_결측,중분류_개수,중분류_결측,공통자체_불일치
지역,,,,,,
강원,77,1,0,3,0,0
경기,943,2,0,6,0,0
경남,73,1,0,3,0,0
경북,558,2,0,6,0,0
광주,287,2,0,5,0,0
대구,232,2,0,6,0,0
대전,260,2,0,6,0,0
부산,570,2,0,6,0,0
서울,184,2,0,5,0,0


In [8]:
# 유틸 함수로 대분류가 전파되지 않은 경우
unsolvedd_major = df_leaf[df_leaf["대분류"].isna()]
print("대분류 미전파 건수: ", len(unsolvedd_major))
print(unsolvedd_major["지역"].value_counts())

display(unsolvedd_major[["지역", "세부사업명", "원본행"]])

대분류 미전파 건수:  0
Series([], Name: count, dtype: int64)


,지역,세부사업명,원본행


## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [9]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
)

display(qa["결과"].value_counts(dropna=False))
display(qa[qa["결과"] == "불일치"])

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

결과
판정불가    89
불일치      2
일치       1
Name: count, dtype: int64

,지역,대분류,중분류,원본_소계값,원본_소계출처,leaf_합계,차이,절대차이,오차율(%),절대오차율(%),QA_병합상태,결과,허용기준결과
21,광주,Ⅱ. 지자체 자체사업,고령화 대책,61202.0,중분류제목행,61201.6,-0.4,0.4,-0.0,0.0,양쪽존재,불일치,허용
65,전남,Ⅱ. 지자체 자체사업,1. 저출산 대책(자체사업),310175.0,중분류제목행,237434.0,-72741.0,72741.0,-23.45,23.45,양쪽존재,불일치,초과


QA 저장 완료: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/reports/2018_전국_QA_검증결과.csv


## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [10]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

유사 사업명 검토 후보: 170


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,경북,1. 저출산 대책(자체사업),임부 초음파 검사비 지원사업,35.0,관내 임부로 보건(지)소 등록시 25천원 상당 초음파쿠폰 2회 지원,임산부 초음파 검사비 지원사업,18.0,초음파검진비지원 1인 3회 지원 (200명×3회),96.774194
1,전남,2. 고령화 대책(자체사업),노인 목욕 및 이미용비 지원,736.0,"70세이상 기초연금수급자에 목욕, 이 미용비 지원(분기당 3매)",노인 목욕비 및 이미용비 지원,1442.0,어르신 이용권 지원,96.774194
2,경기,1. 저출산 대책(자체사업),다자녀가구 상하수도요금 감면,<NA>,"(상수도)최대 10세제곱미터, 10세제곱 미터 미만의 경우엔 실제사용량 감면 (하수도)세자녀-하수도사용요금 20% 네자녀-하수도사용요금 30% 감면",다자녀가구 상수도요금 감면,0.0,다자녀가구 상수도요금 감면,96.551724
3,부산,1. 저출산 대책(자체사업),어린이집 냉·난방비 지원,22.0,어린이집 냉·난방 비용 지원,어린이집 냉난방비 지원,182.0,하․절기 어린이집 냉난방비 지급,96.000000
4,부산,1. 저출산 대책(자체사업),어린이집 냉난방비 지원,16.0,직장어린이집을 제외한 어린이집 보육 현원별 냉난방비 차등 지원,어린이집 냉·난방비 지원,118.0,관내 어린이집을 대상으로 냉·난방비 지원,96.000000
5,부산,1. 저출산 대책(자체사업),어린이집 냉난방비 지원,16.0,직장어린이집을 제외한 어린이집 보육 현원별 냉난방비 차등 지원,어린이집 냉·난방비 지원,22.0,어린이집 냉·난방 비용 지원,96.000000
6,부산,1. 저출산 대책(자체사업),어린이집 냉·난방비 지원,22.0,어린이집 냉·난방 비용 지원,어린이집 냉난방비 지원,84.0,어린이집 냉난방비 지원,96.000000
7,부산,1. 저출산 대책(자체사업),어린이집 냉·난방비 지원,118.0,관내 어린이집을 대상으로 냉·난방비 지원,어린이집 냉난방비 지원,182.0,하․절기 어린이집 냉난방비 지급,96.000000
8,충북,1. 저출산 대책(자체사업),어린이집 아동간식 지원,175.0,학교급식지원사업과 연계하여 어린이집 아동간식 지원,어린이집 아동간식비 지원,110.0,어린이집 아동간식비 지원,96.000000
9,부산,1. 저출산 대책(자체사업),어린이집 냉·난방비 지원,118.0,관내 어린이집을 대상으로 냉·난방비 지원,어린이집 냉난방비 지원,84.0,어린이집 냉난방비 지원,96.000000


## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [11]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸기 전에 `.env`, 설정 파일, 연도별 체크포인트 파일명을 확인한다. 아래 셀은 연결 예시이며, 대량 실행 시 기존 노트북의 청크 단위 체크포인트 저장 로직을 함께 사용한다.

In [12]:
if RUN_LLM:
    import os
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    # 안전 확인용 단건 예시. 대량 처리는 체크포인트 로직을 붙인 뒤 수행한다.
    sample = df_leaf[df_leaf["주요내용"].notna()].iloc[0]
    sample_cleaned = clean_sentence(sample["세부사업명"], sample["주요내용"], call_once=call_once)
    print("원문:", sample["주요내용"])
    print("정제:", sample_cleaned)
    print("숫자보존:", numbers_preserved(sample["주요내용"], sample_cleaned))
    print("재실행대상:", needs_llm_rerun(sample_cleaned))
else:
    print("RUN_LLM=False: LLM 호출을 건너뜁니다.")

원문: 지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강
정제: 지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강
숫자보존: True
재실행대상: False


In [13]:
# 2017 전체 leaf LLM 교정: 병렬 실행, 100건 단위 체크포인트, 숫자 보존 QA
if RUN_LLM:
    from concurrent.futures import ThreadPoolExecutor

    if CHECKPOINT_PATH.exists():
        checkpoint_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig", index_col=0)
        checkpoint_df.index = pd.to_numeric(checkpoint_df.index, errors="raise").astype(int)
        checkpoint_df = checkpoint_df.loc[checkpoint_df.index.intersection(df_leaf.index)]
        rerun_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
        checkpoint_df = checkpoint_df.loc[~rerun_mask].copy()
        print(f"체크포인트: {len(checkpoint_df)}건 유지, {int(rerun_mask.sum())}건 재실행")
    else:
        checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])

    targets = [idx for idx in df_leaf.index if idx not in set(checkpoint_df.index)]
    print(f"LLM 처리 대상: {len(targets)}건 / 전체 {len(df_leaf)}건")

    def clean_target(idx):
        row = df_leaf.loc[idx]
        if pd.isna(row["주요내용"]):
            return idx, None
        # API 실패를 원문 완료로 저장하지 않고 즉시 중단해 다음 실행에서 재개한다.
        cleaned = call_once(row["세부사업명"], str(row["주요내용"]))
        if cleaned is None:
            raise RuntimeError(f"LLM 응답 파싱 실패: {idx}")
        return idx, cleaned

    pending = {}
    with ThreadPoolExecutor(max_workers=1) as executor:
        for processed, (idx, cleaned) in enumerate(executor.map(clean_target, targets), 1):
            pending[idx] = cleaned
            if processed % 100 == 0:
                partial_results = pd.DataFrame.from_dict(
                    pending, orient="index", columns=["주요내용_정제"]
                )
                checkpoint_df = pd.concat([checkpoint_df, partial_results])
                checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
                checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
                pending = {}
                print(f"체크포인트 저장: {processed}/{len(targets)}건")

    if pending:
        partial_results = pd.DataFrame.from_dict(pending, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial_results])
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    checkpoint_df = checkpoint_df.loc[~checkpoint_df.index.duplicated(keep="last")]
    checkpoint_df = checkpoint_df.reindex(df_leaf.index)
    df_leaf["주요내용_정제"] = checkpoint_df["주요내용_정제"]
    # LLM이 다시 만든 독립 불릿은 제거하되 단어 내부 가운데점(교재·교구)은 보존한다.
    df_leaf["주요내용_정제"] = df_leaf["주요내용_정제"].apply(dedup_label)
    checkpoint_df["주요내용_정제"] = df_leaf["주요내용_정제"]
    checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
    df_leaf["숫자보존"] = [
        numbers_preserved(original, cleaned)
        for original, cleaned in zip(df_leaf["주요내용"], df_leaf["주요내용_정제"])
    ]
    bad_index = df_leaf.index[~df_leaf["숫자보존"]]
    if len(bad_index):
        review_cols = ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"]
        df_leaf.loc[bad_index, review_cols].to_csv(
            REPORTS_DIR / f"{YEAR}_LLM_숫자불일치_검토.csv", index=False, encoding="utf-8-sig"
        )
        df_leaf.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]
        checkpoint_df.loc[bad_index, "주요내용_정제"] = df_leaf.loc[bad_index, "주요내용"]
        checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")

    missing_cleaned = df_leaf["주요내용"].notna() & df_leaf["주요내용_정제"].isna()
    df_leaf.loc[missing_cleaned, "주요내용_정제"] = df_leaf.loc[missing_cleaned, "주요내용"]
    print(f"숫자 불일치 원문 복구: {len(bad_index)}건")
    print(f"정제문 결측 원문 복구: {int(missing_cleaned.sum())}건")
else:
    df_leaf["주요내용_정제"] = df_leaf["주요내용"]

체크포인트: 5472건 유지, 0건 재실행
LLM 처리 대상: 0건 / 전체 5472건
숫자 불일치 원문 복구: 0건
정제문 결측 원문 복구: 0건


In [14]:
# 숫자 보존 검사가 놓칠 수 있는 프롬프트 라벨 추가·과도한 축약을 복구한다.
original_text = df_leaf["주요내용"].fillna("").astype(str)
cleaned_text = df_leaf["주요내용_정제"].fillna("").astype(str)
length_ratio = cleaned_text.str.len() / original_text.str.len().replace(0, pd.NA)
added_prompt_label = cleaned_text.str.contains("원문:", regex=False) & ~original_text.str.contains(
    "원문:", regex=False
)
semantic_bad = original_text.ne("") & (
    added_prompt_label | length_ratio.lt(0.5) | length_ratio.gt(1.5)
)
if semantic_bad.any():
    semantic_review = df_leaf.loc[
        semantic_bad,
        ["지역", "원본행", "세부사업명", "주요내용", "주요내용_정제"],
    ].copy()
    semantic_review["길이비"] = length_ratio.loc[semantic_bad]
    semantic_review.to_csv(
        REPORTS_DIR / f"{YEAR}_LLM_의미보존_검토.csv",
        index=False,
        encoding="utf-8-sig",
    )
    df_leaf.loc[semantic_bad, "주요내용_정제"] = df_leaf.loc[semantic_bad, "주요내용"]
    checkpoint_df.loc[semantic_bad, "주요내용_정제"] = df_leaf.loc[semantic_bad, "주요내용"]
    checkpoint_df.to_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
print(f"의미 보존 이상 원문 복구: {int(semantic_bad.sum())}건")

의미 보존 이상 원문 복구: 0건


## 9. 시도별 최종 저장

최종 컬럼은 해당 연도 작업에서 확정한 스키마로 수정한다. 같은 파일을 중간과 최종 단계에서 중복 저장하지 않는다.

In [15]:
df_leaf["연도"] = YEAR

output_cols = [
    "재원구분",
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
# 재원구분은 중복/합산 QA에만 사용하고 최종 저장 스키마에서는 제외한다.
output_cols.remove("재원구분")
output_cols.insert(output_cols.index("주요내용") + 1, "주요내용_정제")
missing_output_cols = set(output_cols).difference(df_leaf.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

for sido, group in df_leaf.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    raw_output_path = sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    df_labeled.loc[df_labeled["지역"].eq(sido)].to_csv(
        raw_output_path, index=False, encoding="utf-8-sig"
    )
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 77행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2018_강원_세부사업_정제.csv
경기: 943행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2018_경기_세부사업_정제.csv
경남: 73행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2018_경남_세부사업_정제.csv
경북: 558행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2018_경북_세부사업_정제.csv
광주: 287행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2018_광주_세부사업_정제.csv
대구: 232행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2018_대구_세부사업_정제.csv
대전: 260행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2018_대전_세부사업_정제.csv
부산: 570행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2018_부산_세부사업_정제.csv
서울: 184행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정저

## 10. Long 포맷 변환 및 저장

당해예산과 전년도예산을 연도별 행으로 변환하고 시도별 파일로 저장한다.

In [16]:
long_id_cols = [column for column in output_cols if column not in {"당해예산", "전년도예산"}]
df_long = df_leaf[output_cols].melt(
    id_vars=long_id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

previous_mask = df_long["예산구분"].eq("전년도예산")
df_long.loc[previous_mask, "연도"] -= 1
df_long.loc[previous_mask, ["증감액", "증감율"]] = None
df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

assert len(df_long) == len(df_leaf) * 2
print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf), "행 x 2)")
display(df_long.head(10))

long 변환 결과: 10944 행 (wide 5472 행 x 2)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,주요내용_정제,증감액,증감율,원본행,지원대상,지원내용_상세,예산구분,예산액
0,2018,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,다함께 돌봄센터 설치,지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강,지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강,100.0,<NA>,5091.0,시군(1개소),돌봄센터 인건비 및 시설 보강,당해예산,100.0
1,2017,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,다함께 돌봄센터 설치,지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강,지원대상 : 시군(1개소) 지원내용 : 돌봄센터 인건비 및 시설 보강,<NA>,<NA>,5091.0,시군(1개소),돌봄센터 인건비 및 시설 보강,전년도예산,0.0
2,2018,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,아이돌봄 서비스 지원,"아이돌봄서비스지원 1500여세대 - 시간제,종일제 유형별 지원","아이돌봄서비스지원 1500여세대 - 시간제,종일제 유형별 지원",757.0,10.4,5094.0,NaN,NaN,당해예산,8067.0
3,2017,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,아이돌봄 서비스 지원,"아이돌봄서비스지원 1500여세대 - 시간제,종일제 유형별 지원","아이돌봄서비스지원 1500여세대 - 시간제,종일제 유형별 지원",<NA>,<NA>,5094.0,NaN,NaN,전년도예산,7310.0
4,2018,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,농촌총각 결혼지원,도내 거주자 중 결혼하지 못한 농촌 총각의 국제결혼 지원,도내 거주자 중 결혼하지 못한 농촌 총각의 국제결혼 지원,-7.0,-11.7,5097.0,NaN,NaN,당해예산,53.0
5,2017,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,농촌총각 결혼지원,도내 거주자 중 결혼하지 못한 농촌 총각의 국제결혼 지원,도내 거주자 중 결혼하지 못한 농촌 총각의 국제결혼 지원,<NA>,<NA>,5097.0,NaN,NaN,전년도예산,60.0
6,2018,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,농어업인 고교생자녀 학자금 지원,교육비 부담이 큰 농업인에게 자녀 학자금을 지원,교육비 부담이 큰 농업인에게 자녀 학자금을 지원,0.0,0.0,5100.0,NaN,NaN,당해예산,1520.0
7,2017,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,농어업인 고교생자녀 학자금 지원,교육비 부담이 큰 농업인에게 자녀 학자금을 지원,교육비 부담이 큰 농업인에게 자녀 학자금을 지원,<NA>,<NA>,5100.0,NaN,NaN,전년도예산,1520.0
8,2018,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,여성농업인 농가도우미 지원,여성농업인의 출산으로 인한 영농공 백시 영농대행을 위한 농가도우미 지원,여성농업인의 출산으로 인한 영농공백시 영농대행을 위한 농가도우미 지원,45.0,60.0,5103.0,NaN,NaN,당해예산,120.0
9,2017,강원,Ⅰ. 공통사업,1. 저출산 대책(공통사업),공통,여성농업인 농가도우미 지원,여성농업인의 출산으로 인한 영농공 백시 영농대행을 위한 농가도우미 지원,여성농업인의 출산으로 인한 영농공백시 영농대행을 위한 농가도우미 지원,<NA>,<NA>,5103.0,NaN,NaN,전년도예산,75.0


In [17]:
for sido, group in df_long.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제_long.csv"
    group.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

강원: 154행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/강원/2018_강원_세부사업_정제_long.csv
경기: 1886행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경기/2018_경기_세부사업_정제_long.csv
경남: 146행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경남/2018_경남_세부사업_정제_long.csv
경북: 1116행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/경북/2018_경북_세부사업_정제_long.csv
광주: 574행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/광주/2018_광주_세부사업_정제_long.csv
대구: 464행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대구/2018_대구_세부사업_정제_long.csv
대전: 520행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/대전/2018_대전_세부사업_정제_long.csv
부산: 1140행 저장 -> /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/data/interim/부산/2018_부산_세부사업_정제_long.csv
서울: 368행 저장 -> /Users

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교